# Qwen3.8-27B reference run on Colab · public_test with Gemini transcripts

The agent of `docker/predict.py` (03c repair's loop: validate first, missing-parameter repair, one retry after an invalid call, empty-search retry, repeat guards) and the prompt of `02_llm_qwen_ver3.ipynb` (`lifemate.py` copied unchanged, week labels included), with the fine-tuned 4B replaced by **`Qwen/Qwen3.8-27B-FP8` zero-shot** in vLLM, thinking on with a token budget. No training, train data is never read.

**Reference only, never the final solution.** 27B breaks the 5B LLM limit, and the final-request text comes from Gemini (an API), so its `submission.jsonl` belongs only where the organisers allow API or oversized runs (the API-test track). Never distill from its public traces: that is labelling test data.

**Setup**
1. Runtime > Change runtime type > **A100 GPU** (high RAM is fine). The first code cell stops on anything under 39 GB.
2. In Google Drive, folder `MyDrive/zola/`:
   - `train+public_test.zip` (the data package; only `public_test` is extracted, without audio)
   - `raw_asr_public.json` (Gemini text, from `D:\ZOLA\asr_compare\`; `D:\transcripts.json`, keyed by sample id, also works)
   - `asr_transcripts.json` (01c output `thbased01`): fills the 1,288 history turns Gemini does not have. The run stops if a clip has no text.
3. Runtime > Run all. The clock starts at the first cell and covers install, the 31 GB download and vLLM start-up.

**5-hour limit**: samples run in chunks of `CHUNK`, all steps of a chunk batched together. No round or chunk starts within `MARGIN_MIN` of the limit. Samples never started are written as empty answers (stop `not_run`).

**Output** in `MyDrive/zola/qwen38_out/` (Drive, so a disconnect keeps it): `traces.jsonl` (calls, hash, stop, each step's thought and answer, request text) and `submission.jsonl`, validated by `zaic_tools`. **To continue** after a disconnect or the limit, run all again: finished samples are kept, `not_run` ones run. `ckpt.json` holds every sample still running after each round (its calls, conversation and steps), so after a disconnect those samples replay their calls on a new env (the old one died with the runtime) and continue from the last round. Only the round running at the disconnect is redone. Stopping the cell yourself ends the running samples as `crash` instead.

**If vLLM runs out of memory at start-up** (40 GB A100: about 29 GB of weights): `LLM_KWARGS = {"enforce_eager": True}`, then `"max_num_batched_tokens": 4096`. If `language_model_only` is rejected: `{"language_model_only": None}`. Speed on an A100 is not measured, so watch the minutes-left line after the first chunk.

In [ ]:
import time

T0 = time.time()  # the 5-hour clock: install, model download and vLLM start-up count too

HOURS = 5.0  # whole notebook
MARGIN_MIN = 20  # no round or chunk starts this close to the end: one round (minutes on a 40 GB A100) plus the submission must still fit
DRIVE = "/content/drive/MyDrive/ZOLA_data"
ZIP = f"{DRIVE}/train+public_test.zip"
GEMINI = f"{DRIVE}/raw_asr_public.json"  # Gemini text: keys "<split>/<clip path>", or sample ids with <id>_h<N> for history[N]
FILL = f"{DRIVE}/asr_transcripts.json"  # 01c output: every clip, used where Gemini has none
SPLIT = "public_test"
MODEL_ID = "Qwen/Qwen3.8-27B-FP8"  # official block FP8, 30.9 GB; weight-only Marlin kernels on an A100
THINK = True  # False: empty think block, answers only
EFFORT = "low"  # Qwen3.8 chat template: "low" asks for brief thinking, "medium", "xhigh" (its default)
THINK_TOKENS = 1024  # per step; a longer thought is closed for the model, then the answer is generated
ANSWER_TOKENS = 320  # the longest gold call is 304 characters
MAX_STEPS = 8  # gold never uses more than 6 calls
CHUNK = 64  # samples batched together
MAX_LEN = 16384  # the competition's context limit
LLM_KWARGS = {}  # calibration knob for vLLM's LLM(...), e.g. {"enforce_eager": True}; None removes a default
SEED = 42

import os

from google.colab import drive

drive.mount("/content/drive")  # first, so the sign-in prompt comes before the long install
assert os.path.exists(ZIP) and os.path.exists(GEMINI), f"put train+public_test.zip and raw_asr_public.json in {DRIVE}"

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
print(gpu)
assert gpu and int(gpu.split(",")[1]) >= 39000, "needs an A100 (40 or 80 GB): Runtime > Change runtime type > A100 GPU"
!pip install -q vllm==0.32.0
assert _exit_code == 0, "vllm install failed"

In [ ]:
%%writefile lifemate.py
"""LifeMate shared code: data split, prompt format, action parser, scorer.
02 trains on this file and saves it next to the model; 03 imports that copy, so the prompt always matches the weights.
Self-check: python lifemate.py <data root>"""
import collections
import json
import random
import re
import sys
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path

ASK = "sys.ask_missing_info"
BOS, STOP = "", "<|im_end|>"
ROLES = {r: f"<|im_start|>{r}\n" for r in ("system", "user", "assistant")}  # plain ChatML: Qwen templates add a think block only to the last assistant turn, which would make training and inference differ
GEN = ROLES["assistant"]
MAX_RESULT_CHARS = 1500  # ponytail: hard cut on long tool results, summarise per tool if truncation ever hides an ID
HOLDOUT_APPS = {"door_lock", "learning", "library", "loyalty", "subscriptions", "household_chores", "home_security"}  # train-only apps: stands in for the unseen apps of the test splits
RELATIVE = {0: "hôm nay", 1: "ngày mai", 2: "ngày kia"}
WEEKS = ["tuần này", "tuần sau", "tuần sau nữa"]  # Monday-start weeks: train gold reads "thứ bảy tuần sau" as next week's Saturday
WEEKDAYS = ["Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy", "Chủ Nhật"]
RULES = """Bạn là trợ lý giọng nói tiếng Việt. Hãy xử lý yêu cầu CUỐI CÙNG của người dùng bằng cách gọi tool, người dùng có thể đổi ý giữa chừng.
Mỗi lượt trả về đúng một dòng JSON, không viết gì khác:
- Gọi tool: {"name": "<app.tool>", "arguments": {...}}
- Kết thúc: {"done": true}
Quy tắc:
- Chỉ gọi tool trong danh sách, đúng tên và đúng kiểu tham số. Không tự đoán ID, hãy đọc bằng tool trước khi ghi. ID đã có trong hội thoại thì dùng lại, không đọc lại.
- Chỉ gọi tool thật sự cần. Đọc xong mà yêu cầu cần ghi thì phải ghi tiếp, đừng dừng ở bước đọc.
- Tham số không bắt buộc: chỉ điền khi người dùng nói rõ, không tự điền giá trị mặc định. Tool đọc không thêm bộ lọc nếu người dùng không nêu.
- Yêu cầu có điều kiện ("nếu mai mưa thì..."): đọc điều kiện trước, chỉ làm nhánh đúng. Điều kiện sai thì {"done": true}.
- Tìm thấy đúng một kết quả phù hợp thì dùng luôn. Không tìm thấy, nhiều kết quả không phân biệt được, hoặc thiếu tham số bắt buộc: làm xong các việc khác rồi gọi sys.ask_missing_info với tool_name là tool định gọi và params là tên các tham số còn thiếu, sau đó kết thúc.
- Không app nào làm được, người dùng tự huỷ ("à thôi khỏi"), phủ định ("đừng tắt"), hoặc chỉ trò chuyện: {"done": true} ngay, không gọi tool.
- Thời gian dùng ISO-8601 đủ giây và múi giờ, ví dụ 2026-11-11T05:30:00+07:00. Tra ngày trong bảng 14 ngày tới, không tự tính.
- Số tiền là số nguyên đồng (2 triệu = 2000000). Số đọc từng chữ số thì chép đúng thứ tự, số điện thoại giữ dạng chuỗi.
- Văn bản tự do (tin nhắn, tiêu đề, ghi chú): viết lại ngắn gọn ý chính, giữ dấu tiếng Việt, tên riêng và con số. Từ khoá tìm kiếm chỉ là tên hoặc cụm từ người dùng nhắc."""


def load_split(root, split):
    with open(Path(root) / split / f"{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def load_catalog(root, split):
    with open(Path(root) / split / f"{split}_tools_catalog.json", encoding="utf-8") as f:
        return {t["name"]: t for a in json.load(f)["apps"] for t in a["tools"]}


def used_apps(x):
    return {(c["arguments"]["tool_name"] if c["name"] == ASK else c["name"]).split(".")[0] for c in x["label"]["tool_calls"]}


def clips(x):
    """Audio paths of a sample: the final request plus every user turn of the history."""
    return [x["audio"]] + [t["audio"] for t in x["history"] if t["role"] == "user"]


def gold_texts(x):
    return {x["audio"]: x["audio_text"]} | {t["audio"]: t["audio_text"] for t in x["history"] if t["role"] == "user"}


def split_rows(rows):
    """(train, val_unseen_apps, val_seen_apps). Final run: train on all rows instead."""
    train, unseen, seen = [], [], []
    for i, x in enumerate(rows):
        (unseen if used_apps(x) & HOLDOUT_APPS else seen if i % 12 == 0 else train).append(x)
    return train, unseen, seen


def render_tool(t):
    """Keeps kind, returns and x-ref: they say what is a read and which read supplies an ID, which an unseen app needs."""
    p = t["parameters"]
    o = {"name": t["name"], "kind": t["kind"], "description": t["description"]}
    if p.get("properties"):
        o["parameters"] = {k: {a: b for a, b in v.items() if a != "x-match"} for k, v in p["properties"].items()}
    for k in ("required", "x-one-of"):
        if p.get(k):
            o[k] = p[k]
    if t.get("returns"):
        o["returns"] = t["returns"]
    return json.dumps(o, ensure_ascii=False, separators=(",", ":"))


def system_prompt(ctx, tools):
    """Context stays in the system prompt, not beside the request as in Spark ver3: history calls are training targets too, and they need the clock."""
    now = datetime.fromisoformat(ctx["current_time"])
    days = ", ".join(f"{WEEKDAYS[d.weekday()]} {d:%Y-%m-%d} ({', '.join(filter(None, [RELATIVE.get(i), WEEKS[(now.weekday() + i) // 7]]))})" for i, d in enumerate(now + timedelta(i) for i in range(14)))
    return (f"{RULES}\n\nBây giờ: {ctx['current_time']} ({WEEKDAYS[now.weekday()]})\n14 ngày tới: {days}\n"
            f"Vị trí: {json.dumps(ctx.get('location'), ensure_ascii=False)}\n"
            f"Người dùng: {json.dumps(ctx.get('user_profile'), ensure_ascii=False)}\n\nDanh sách tool:\n" + "\n".join(render_tool(t) for t in tools))


def asst(obj):
    return {"role": "assistant", "content": json.dumps(obj, ensure_ascii=False)}


def result_msg(result):
    return {"role": "user", "content": "KẾT QUẢ: " + json.dumps(result, ensure_ascii=False)[:MAX_RESULT_CHARS]}


def build_messages(item, texts, catalog, rng=None):
    """texts: {audio path: transcript}. rng shuffles the tools, so the model cannot learn a position. Roles strictly alternate."""
    ctx = item["context"]
    names = [t for a in ctx["installed_apps"].values() for t in a["tools"]] + [ASK]
    if rng:
        rng.shuffle(names)
    msgs = [{"role": "system", "content": system_prompt(ctx, [catalog[n] for n in names])}]
    for turn in item["history"]:
        if turn["role"] == "user":
            msgs.append({"role": "user", "content": texts[turn["audio"]]})
            continue
        results = turn.get("tool_results") or []
        for i, c in enumerate(turn.get("tool_calls") or []):
            msgs += [asst({"name": c["name"], "arguments": c["arguments"]}), result_msg(results[i] if i < len(results) else {"ok": True})]
        msgs.append(asst({"done": True, "text": turn.get("text", "")}))
    msgs.append({"role": "user", "content": texts[item["audio"]]})
    return msgs


def gold_steps(x):
    """Gold trajectory after the final user turn: call, result, ..., done."""
    msgs = []
    for c, r in zip(x["label"]["tool_calls"], x["label"]["tool_results"]):
        msgs += [asst({"name": c["name"], "arguments": c["arguments"]}), result_msg(r)]
    return msgs + [asst({"done": True})]


def parse_action(text):
    """(name, arguments), or None for done / unparsable output (both end the turn)."""
    try:
        o = json.JSONDecoder().raw_decode(text[text.index("{"):])[0]
    except ValueError:
        return None
    if not isinstance(o.get("name"), str):
        return None
    return o["name"], o["arguments"] if isinstance(o.get("arguments"), dict) else {}


def tokens(s):
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFC", str(s).lower()))  # word tokens, so "nhé." equals "nhé". The official tokeniser is unconfirmed


def token_f1(a, b):
    a, b = tokens(a), tokens(b)
    c = sum((collections.Counter(a) & collections.Counter(b)).values())
    return 2 * c / (len(a) + len(b)) if c else 0.0


def same(a, b):
    return json.dumps(a, sort_keys=True) == json.dumps(b, sort_keys=True)  # type-strict: 1, 1.0 and true all differ


def call_match(p, g, catalog):
    pa, ga = p.get("arguments") or {}, g.get("arguments") or {}
    if p["name"] != g["name"] or set(pa) != set(ga):
        return False
    props = catalog.get(g["name"], {}).get("parameters", {}).get("properties", {})
    return all(token_f1(pa[k], v) > 0.6 if props.get(k, {}).get("x-match") == "fuzzy" else same(pa[k], v) for k, v in ga.items())


def matches(pred, gold, catalog):
    """Size of the maximum one-to-one matching (Kuhn), which a greedy pass can undercount."""
    edges = [[j for j, g in enumerate(gold) if call_match(p, g, catalog)] for p in pred]
    owner = {}

    def augment(i, seen):
        for j in edges[i]:
            if j not in seen:
                seen.add(j)
                if j not in owner or augment(owner[j], seen):
                    owner[j] = i
                    return True
        return False

    return sum(augment(i, set()) for i in range(len(pred)))


def score(pred, gold, pred_hash, gold_hash, catalog):
    """Official formula from the description."""
    if not gold:
        return float(not pred)
    if pred_hash != gold_hash:
        return 0.0
    m = matches(pred, gold, catalog)
    if not m:
        return 0.0
    precision, recall = m / len(pred), m / len(gold)
    return 2 * precision * recall / (precision + recall)


def render(msgs):
    return "".join(f"{ROLES[m['role']]}{m['content']}{STOP}\n" for m in msgs)


def prompt_text(msgs):
    """Inference prompt, rendered here so that training and inference always see the same text."""
    return BOS + render(msgs) + GEN


def targets(msgs, first_target, history_calls):
    """Indices of the assistant turns to learn: every turn from first_target, plus the tool calls of history turns if history_calls.
    History replies ({"done": true, "text": ...}) stay unlabelled, since the final turn never writes text."""
    return {i for i, m in enumerate(msgs) if m["role"] == "assistant" and (i >= first_target or history_calls and m["content"].startswith('{"name"'))}


def mask_apps(msgs, apps, rng):
    """Training only: every installed app gets a made-up name in every turn, so the model has to read the tool list instead of
    remembering app names (function masking, as in Hammer, arXiv 2410.04587). Train results and transcripts never hold `app.`, checked."""
    new = {a: "".join(rng.choices("abcdefghijklmnopqrstuvwxyz", k=rng.randint(4, 9))) for a in apps}
    pat = re.compile(rf"(?<![\w@.])({'|'.join(map(re.escape, new))})\.(?=[a-z_])")
    return [m | {"content": pat.sub(lambda g: new[g.group(1)] + ".", m["content"])} for m in msgs]


def encode(tok, msgs, labelled):
    """Tokenise one conversation, labelling only the assistant turns whose index is in labelled."""
    text, spans = BOS, []
    for i, m in enumerate(msgs):
        start = len(text) + len(ROLES[m["role"]])
        text += render([m])
        if i in labelled:
            spans.append((start, len(text) - 1))  # content + end token, without the newline
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    labels = [t if any(a <= s and e <= b for a, b in spans) else -100 for t, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


def demo(root):
    sys.path.insert(0, str(Path(root) / "train"))
    from zaic_tools import ToolEnv

    rows, catalog = load_split(root, "train"), load_catalog(root, "train")
    for x in rows[:200]:  # gold replay must score full marks, otherwise the scorer or the data path is broken
        env = ToolEnv("train", x["id"], root=root)
        gold = x["label"]["tool_calls"]
        results = [env.call(c["name"], c["arguments"]) for c in gold]
        assert results == x["label"]["tool_results"], x["id"]
        assert score(env.get_tool_calls(), gold, "h", "h", catalog) == 1.0, x["id"]

    g = [{"name": "sms.send", "arguments": {"contact_id": "c_1", "content": "mai họp lúc chín giờ"}}]
    near = [{"name": "sms.send", "arguments": {"contact_id": "c_1", "content": "Mai họp lúc chín giờ nhé"}}]
    extra = g + [{"name": "contacts.search", "arguments": {"query": "x"}}]
    wrong_id = [{"name": "sms.send", "arguments": {"contact_id": "c_2", "content": "mai họp lúc chín giờ"}}]
    assert score(near, g, "a", "a", catalog) == 1.0
    assert score(g, g, "a", "b", catalog) == 0.0
    assert score(wrong_id, g, "a", "a", catalog) == 0.0
    assert score([], g, "a", "a", catalog) == 0.0
    assert score(g, [], "a", "a", catalog) == 0.0
    assert score([], [], "x", "y", catalog) == 1.0
    assert abs(score(extra, g, "a", "a", catalog) - 2 / 3) < 1e-9
    assert token_f1("mai họp nhé.", "Mai họp, nhé") == 1.0
    fz = {"t.x": {"parameters": {"properties": {"q": {"x-match": "fuzzy"}}}}}
    assert not call_match({"name": "t.x", "arguments": {"q": "a b", "n": 1}}, {"name": "t.x", "arguments": {"q": "a b"}}, fz)  # extra key
    assert not call_match({"name": "t.x", "arguments": {"n": 1}}, {"name": "t.x", "arguments": {"n": True}}, fz)  # type
    q = [{"name": "t.x", "arguments": {"q": s}} for s in ("một hai ba bốn", "hai ba năm sáu", "một hai ba năm")]
    assert matches(q[:2], [q[2], q[0]], fz) == 2  # the first pred fits both golds, the second only the first: greedy finds 1

    assert parse_action('{"name": "a.b", "arguments": {"x": 1}} rác') == ("a.b", {"x": 1})
    assert parse_action('rác {"name": "a.b"}') == ("a.b", {})
    assert parse_action('{"done": true}') is None
    assert parse_action("không có json") is None
    assert parse_action('{"name": ') is None

    sp = system_prompt({"current_time": "2026-09-24T09:00:00+07:00"}, [])  # a Thursday
    assert "Thứ Năm 2026-09-24 (hôm nay, tuần này)" in sp and "Chủ Nhật 2026-09-27 (tuần này)" in sp and "Thứ Bảy 2026-10-03 (tuần sau)" in sp, sp

    train, unseen, seen = split_rows(rows)
    assert len(train) + len(unseen) + len(seen) == len(rows)
    assert not any(used_apps(x) & HOLDOUT_APPS for x in train)
    for x in rows:
        msgs = build_messages(x, gold_texts(x), catalog) + gold_steps(x)
        roles = [m["role"] for m in msgs[1:]]
        assert all(a != b for a, b in zip(roles, roles[1:])), x["id"]
        first = len(msgs) - 2 * len(x["label"]["tool_calls"]) - 1
        assert msgs[first - 1]["content"] == x["audio_text"], x["id"]
        hist = [c for t in x["history"] for c in t.get("tool_calls") or []]
        assert len(targets(msgs, first, True)) == len(hist) + len(x["label"]["tool_calls"]) + 1 and len(targets(msgs, first, False)) == len(x["label"]["tool_calls"]) + 1, x["id"]
        m = mask_apps(msgs, x["context"]["installed_apps"], random.Random(0))
        called = [json.loads(t["content"])["name"] for t in m[first:] if t["role"] == "assistant" and t["content"].startswith('{"name"')]
        assert all(f'"name":"{n}"' in m[0]["content"] for n in called) and not any(f'"name":"{a}.' in m[0]["content"] for a in x["context"]["installed_apps"]), x["id"]
    print("lifemate self-check ok:", len(train), "train,", len(unseen), "unseen-app val,", len(seen), "seen-app val")


if __name__ == "__main__":
    demo(sys.argv[1])

In [ ]:
import glob
import json
import sys
import zipfile
from collections import Counter
from pathlib import Path

DATA, OUT = Path("/content/data"), Path(DRIVE) / "qwen38_out"
OUT.mkdir(parents=True, exist_ok=True)
if not glob.glob(f"{DATA}/**/{SPLIT}/{SPLIT}.jsonl", recursive=True):
    with zipfile.ZipFile(ZIP) as z:  # this split only, no audio: the agent reads transcripts
        z.extractall(DATA, [m for m in z.namelist() if f"/{SPLIT}/" in "/" + m and not m.endswith(".wav")])
ROOT = Path(glob.glob(f"{DATA}/**/{SPLIT}/{SPLIT}.jsonl", recursive=True)[0]).parents[1]
sys.path[:0] = [".", str(ROOT / SPLIT)]  # lifemate.py from this notebook, zaic_tools of this split
import lifemate as lm
from zaic_tools import ToolEnv

catalog, rows = lm.load_catalog(ROOT, SPLIT), lm.load_split(ROOT, SPLIT)


def load_texts(path):
    """{"<split>/<clip path>": text}. Also takes transcripts keyed by sample id: <id> is the final request, <id>_h<N> is history[N]."""
    t = json.load(open(path, encoding="utf-8"))
    if all("/" in k for k in t):
        return t
    clip = {x["id"]: x["audio"] for x in rows} | {f"{x['id']}_h{n}": h["audio"] for x in rows for n, h in enumerate(x["history"]) if h["role"] == "user"}
    return {f"{SPLIT}/{clip[k]}": v for k, v in t.items() if k in clip}


gem = load_texts(GEMINI)
fill = load_texts(FILL) if os.path.exists(FILL) else {}
texts = fill | gem  # Gemini wins wherever it has the clip
need = [f"{SPLIT}/{c}" for x in rows for c in lm.clips(x)]
miss = [k for k in need if k not in texts]
print(f"{ROOT}: {len(rows)} samples, {len(need)} clips: {sum(k in gem for k in need)} Gemini, {sum(k not in gem and k in fill for k in need)} 01c, {len(miss)} missing")
assert not miss, f"no text for {len(miss)} clips, e.g. {miss[:3]}: put 01c's asr_transcripts.json at {FILL}"

In [ ]:
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # engine in this process, as in docker/predict.py
from vllm import LLM, SamplingParams

kw = dict(model=MODEL_ID, seed=SEED, max_model_len=MAX_LEN, max_num_seqs=32, gpu_memory_utilization=0.93, language_model_only=True,
          generation_config="vllm")  # ignore the repo's sampling defaults; every request below is greedy
llm = LLM(**{k: v for k, v in (kw | LLM_KWARGS).items() if v is not None})
tok = llm.get_tokenizer()
STOP = tok.encode(lm.STOP, add_special_tokens=False)
END = tok.convert_tokens_to_ids("</think>")
CLOSE = tok.encode("\n</think>\n\n", add_special_tokens=False)
assert len(STOP) == 1 and tok.encode("</think>", add_special_tokens=False) == [END], (STOP, END)


def params(n):
    return SamplingParams(temperature=0, max_tokens=n, stop_token_ids=STOP, skip_special_tokens=False)  # keep </think> in the text


def prompt_ids(msgs):
    """Qwen3.8's own chat template, so thinking and reasoning_effort work as the model was trained."""
    return tok.encode(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=THINK, reasoning_effort=EFFORT), add_special_tokens=False)


def generate(convs):
    """[(thought, answer, capped)] per conversation, one vLLM batch. A thought cut at THINK_TOKENS is closed for the model, then answered."""
    ids = [prompt_ids(m) for m in convs]
    budget = THINK_TOKENS if THINK else ANSWER_TOKENS
    outs = [o.outputs[0] for o in llm.generate([{"prompt_token_ids": i} for i in ids], [params(max(1, min(budget, MAX_LEN - len(i) - ANSWER_TOKENS - 8))) for i in ids], use_tqdm=False)]
    texts, capped = [o.text for o in outs], [0] * len(outs)
    again = [k for k, o in enumerate(outs) if THINK and o.finish_reason == "length"]
    if again:  # token ids, not text, so the continuation sees exactly what was generated
        cut = [END not in outs[k].token_ids for k in again]
        ext = [ids[k] + list(outs[k].token_ids) + (CLOSE if c else []) for k, c in zip(again, cut)]
        for k, c, o in zip(again, cut, llm.generate([{"prompt_token_ids": e} for e in ext], params(ANSWER_TOKENS), use_tqdm=False)):
            texts[k] += ("\n</think>\n\n" if c else "") + o.outputs[0].text
            capped[k] = int(c)
    return [(*t.rpartition("</think>")[::2], c) for t, c in zip(texts, capped)]


sample = lm.build_messages(rows[0], {c: texts[f"{SPLIT}/{c}"] for c in lm.clips(rows[0])}, catalog)
print(tok.apply_chat_template(sample, tokenize=False, add_generation_prompt=True, enable_thinking=THINK, reasoning_effort=EFFORT)[-400:])
print(f"model ready after {(time.time() - T0) / 60:.1f} min, longest first prompt {max(len(prompt_ids(lm.build_messages(x, {c: texts[f'{SPLIT}/{c}'] for c in lm.clips(x)}, catalog))) for x in rows)} tokens")

In [ ]:
def missing(name, args, installed):
    """The ask gold would make for an installed tool that misses a required or one-of parameter, else None. From docker/predict.py."""
    tool = catalog.get(name)
    if tool is None or name.split(".")[0] not in installed:
        return None
    p = tool.get("parameters", {})
    empty = lambda k: args.get(k) in (None, "", [], {})  # zaic_tools counts these as missing
    need = [k for k in p.get("required", []) if empty(k)]
    if p.get("x-one-of") and all(any(empty(k) for k in grp) for grp in p["x-one-of"]):
        need.append(p["x-one-of"][0][0])
    return {"tool_name": name, "params": need} if need else None


def step(s, answer):
    """One model answer on the sample's env; False when the sample stops. The branches of docker/predict.py's run()."""
    env, msgs = s["env"], s["msgs"]
    s["raw"].append(answer)
    action = lm.parse_action(answer)
    if action is None:  # parse_action returns None for both
        s["stop"] = "done" if '"done"' in answer else "unparsable"
        return False
    call = {"name": action[0], "arguments": action[1]}
    said = json.JSONDecoder().raw_decode(answer[answer.index("{"):])[0]
    check = env.validate(*action)
    if not check["ok"]:  # validate is not logged: a call we never make costs no U penalty
        ask = missing(*action, s["item"]["context"]["installed_apps"])
        if ask and env.validate(lm.ASK, ask)["ok"]:
            env.call(lm.ASK, ask)
            s["stop"] = "repaired"
            return False
        if not s["retried"]:  # one more try with the error as the result
            s["retried"] = True
            msgs += [lm.asst(said), lm.result_msg(check)]
            return True
        s["stop"] = "invalid"
        return False
    calls = env.get_tool_calls()
    if calls.count(call) >= 2 or catalog.get(call["name"], {}).get("kind") == "read" and [c["name"] for c in calls].count(call["name"]) >= 2:
        s["stop"] = "repeat"
        return False
    result = env.call(*action)
    msgs += [lm.asst(said), lm.result_msg(result)]
    if "query" in action[1] and not catalog[call["name"]]["parameters"].get("required") and any(v == [] for v in result.values()):  # misheard query: list everything once
        msgs += [lm.asst({"name": call["name"], "arguments": {}}), lm.result_msg(env.call(call["name"], {}))]
    if call["name"] == lm.ASK:  # gold always ends here
        s["stop"] = "ask"
        return False
    return True


def new(x):
    s = {"item": x, "env": ToolEnv(SPLIT, x["id"], root=ROOT), "msgs": lm.build_messages(x, {c: texts[f"{SPLIT}/{c}"] for c in lm.clips(x)}, catalog),
         "raw": [], "thoughts": [], "capped": 0, "stop": "max_steps", "retried": False, "saved": False}  # exactly one env per sample
    c = resume.pop(x["id"], None)
    if c:  # mid-sample at the last checkpoint: its env died with the runtime, so replay its calls on the new one and carry on
        for call in c["calls"]:
            s["env"].call(call["name"], call["arguments"])
        assert s["env"].get_tool_calls() == c["calls"], x["id"]
        s |= {k: c[k] for k in ("msgs", "raw", "thoughts", "capped", "retried")}
    return s


def checkpoint(live):
    """The samples still running, after every round, so a disconnect loses no step. Written to a temp file first: a cut write leaves the old one."""
    tmp = CKPT.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump([{"id": s["item"]["id"], "calls": s["env"].get_tool_calls()} | {k: s[k] for k in ("msgs", "raw", "thoughts", "capped", "retried")} for s in live], f, ensure_ascii=False)
    os.replace(tmp, CKPT)


def save(s, f):
    if s["saved"]:
        return
    env, x = s["env"], s["item"]
    f.write(json.dumps({"id": x["id"], "tool_calls": env.get_tool_calls(), "state_hash": env.get_state_hash(), "stop": s["stop"], "capped": s["capped"],
                        "raw": s["raw"], "thoughts": s["thoughts"], "request": texts[f"{SPLIT}/{x['audio']}"]}, ensure_ascii=False) + "\n")
    f.flush()
    s["saved"] = True


TRACES = OUT / "traces.jsonl"
old = {}
if TRACES.exists():
    for line in map(json.loads, open(TRACES, encoding="utf-8")):
        old[line["id"]] = line  # the last line of a sample wins
todo = [x for x in rows if old.get(x["id"], {}).get("stop", "not_run") == "not_run"]
CKPT = OUT / "ckpt.json"
resume = {c["id"]: c for c in (json.load(open(CKPT, encoding="utf-8")) if CKPT.exists() else []) if c["id"] in {x["id"] for x in todo}}
todo.sort(key=lambda x: x["id"] not in resume)  # checkpointed samples go in the first chunk
print(f"{len(resume)} samples resume from {CKPT}")
left = lambda: T0 + (HOURS * 60 - MARGIN_MIN) * 60 - time.time()
print(f"{len(rows) - len(todo)} done in earlier runs, {len(todo)} to run, {left() / 60:.0f} min before the margin")
with open(TRACES, "a", encoding="utf-8") as f:
    for start in range(0, len(todo), CHUNK):
        if left() <= 0:
            print("5-hour limit: the rest become empty answers below; run all again to continue")
            break
        t, live = time.time(), [new(x) for x in todo[start:start + CHUNK]]
        try:
            while live and left() > 0:
                going = []
                for s, (thought, answer, capped) in zip(live, generate([s["msgs"] for s in live])):
                    s["thoughts"].append(thought)
                    s["capped"] += capped
                    try:
                        on = step(s, answer)
                    except Exception as e:  # keep the calls already made
                        s["stop"], on = "crash", False
                        print(s["item"]["id"], repr(e), flush=True)
                    if on and len(s["raw"]) < MAX_STEPS:  # steps count across a resume
                        going.append(s)
                    else:
                        save(s, f)
                live = going
                checkpoint(live)
            if left() <= 0:
                for s in live:
                    s["stop"] = "deadline"
        except BaseException:  # an engine error or a stopped cell: the calls made so far still go out
            for s in live:
                s["stop"] = "crash"
            raise
        finally:
            for s in live:
                save(s, f)
            checkpoint([])
        print(f"{min(start + CHUNK, len(todo))}/{len(todo)}: chunk {(time.time() - t) / 60:.1f} min, {left() / 60:.0f} min before the margin", flush=True)

In [ ]:
by_id = {}
for line in map(json.loads, open(TRACES, encoding="utf-8")):
    by_id[line["id"]] = line
with open(TRACES, "a", encoding="utf-8") as f:
    for x in rows:
        if x["id"] not in by_id:  # never started: an empty answer on an env with no call, so a later run may still do it
            by_id[x["id"]] = {"id": x["id"], "tool_calls": [], "state_hash": ToolEnv(SPLIT, x["id"], root=ROOT).get_state_hash(), "stop": "not_run"}
            f.write(json.dumps(by_id[x["id"]]) + "\n")
lines = [by_id[x["id"]] for x in rows]
thoughts = [t for x in lines for t in x.get("thoughts", [])]
print("stops:", Counter(x["stop"] for x in lines).most_common())
print(f"{sum(len(x['tool_calls']) for x in lines)} calls, {len(thoughts)} model steps, {sum(map(len, thoughts)) / max(1, len(thoughts)):.0f} thought chars per step, "
      f"{sum(x.get('capped', 0) for x in lines)} thoughts cut at THINK_TOKENS")
SUB = OUT / "submission.jsonl"
with open(SUB, "w", encoding="utf-8") as f:
    for x in lines:
        f.write(json.dumps({k: x[k] for k in ("id", "tool_calls", "state_hash")}, ensure_ascii=False) + "\n")
!cd "{ROOT}/{SPLIT}" && PYTHONPATH=. python -m zaic_tools --split {SPLIT} --root .. --validate "{SUB}"
print(f"{SUB}, {(time.time() - T0) / 3600:.2f} h in total")